In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import seaborn as sns
import torch
import torch.nn as nn
import torchmetrics
import torchvision
import torchvision.transforms as T
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from torch_uncertainty.losses import ELBOLoss
from torch_uncertainty.models.classification.lenet import bayesian_lenet

%matplotlib inline
sns.set_theme(style="whitegrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# MNIST vs. Fashion-MNIST

## Загрузка датасетов

In [ ]:
mnist_transforms = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])

mnist_train = torchvision.datasets.MNIST(
    root="../data", train=True, download=True, transform=mnist_transforms
)
mnist_train_loader = DataLoader(
    mnist_train, batch_size=128, shuffle=True, num_workers=2
)

mnist_test = torchvision.datasets.MNIST(
    root="../data", train=False, download=True, transform=mnist_transforms
)
mnist_id_loader = DataLoader(
    mnist_test, batch_size=128, shuffle=False, num_workers=2
)

fmnist_test = torchvision.datasets.FashionMNIST(
    root="../data", train=False, download=True, transform=mnist_transforms
)
fmnist_ood_loader = DataLoader(
    fmnist_test, batch_size=128, shuffle=False, num_workers=2
)

## Обучение байесовской модели

In [ ]:
# pytorch_lightning и torch-uncertainty обожают выводить
# в консоль непрошенные советы, так что глушим их
warnings.filterwarnings("ignore")

logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)


class LitBayesianLeNet(pl.LightningModule):
    def __init__(self):
        super().__init__()

        self.model = bayesian_lenet(
            in_channels=1, num_classes=10, num_samples=20
        )
        self.acc_metric = torchmetrics.Accuracy(
            task="multiclass", num_classes=10
        )
        self.loss_fn = ELBOLoss(
            model=self.model,
            inner_loss=nn.CrossEntropyLoss(),
            kl_weight=1.0 / 10_000,
            num_samples=3,
        )

    def eval_forward(self, x):
        return self.model.eval_forward(x)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch):
        x, y = batch

        logits = self(x)
        acc = self.acc_metric(logits, y)
        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

        loss = self.loss_fn(x, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=2e-2)


bbb_model = LitBayesianLeNet().to(device)

trainer = pl.Trainer(
    max_epochs=3,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
    enable_progress_bar=True,
)
trainer.fit(bbb_model, mnist_train_loader)

# После обучения модель перекидывается на cpu
bbb_model.to(device).eval();

## Получение неопределенностей и визуализация

In [ ]:
@torch.no_grad()
def get_uncertainties(model, dataloader, num_passes):
    total, aleatoric, epistemic = [], [], []

    for images, _ in dataloader:
        images = images.to(device)

        outputs = model.eval_forward(images).view(
            num_passes, images.size(0), -1
        )
        probs = torch.softmax(outputs, dim=-1)

        p_mean = probs.mean(dim=0)

        total_uncertainty = -torch.sum(
            p_mean * torch.log(p_mean + 1e-12), dim=-1
        )
        aleatoric_uncertainty = -torch.sum(
            probs * torch.log(probs + 1e-12), dim=-1
        ).mean(dim=0)
        epistemic_uncertainty = total_uncertainty - aleatoric_uncertainty

        total.append(total_uncertainty.cpu().numpy())
        aleatoric.append(aleatoric_uncertainty.cpu().numpy())
        epistemic.append(epistemic_uncertainty.cpu().numpy())

    return (
        np.concatenate(total),
        np.concatenate(aleatoric),
        np.concatenate(epistemic),
    )


total_id, aleatoric_id, epistemic_id = get_uncertainties(
    bbb_model, mnist_id_loader, 20
)

total_ood, aleatoric_ood, epistemic_ood = get_uncertainties(
    bbb_model, fmnist_ood_loader, 20
)

In [ ]:
y_true = np.concatenate(
    [np.ones_like(epistemic_ood), np.zeros_like(epistemic_id)]
)
auroc_total = roc_auc_score(y_true, np.concatenate([total_ood, total_id]))

plt.figure(figsize=(6, 5))

sns.kdeplot(
    total_id, fill=True, color="#3498db", label="MNIST (ID)", clip=(0, None)
)
sns.kdeplot(
    total_ood,
    fill=True,
    color="#e74c3c",
    label="Fashion-MNIST (OOD)",
    clip=(0, None),
)

plt.title(f"Полная неопределенность\nAUROC = {auroc_total:.3f}", fontsize=14)
plt.xlabel("Значение энтропии", fontsize=12)
plt.ylabel("Плотность", fontsize=12)
plt.legend()

plt.tight_layout()
plt.show()

В целом видно, что можно с хорошей точностью отделить ID от OOD: в первом случае острый пик в нуле (для большинства изображений неопределенность близка к 0), а во втором -- пик находится примерно в 2.2 (OOD изображения имеют общую неопределенность выше). Посмотрим, на сооотношение алеаторной и эпистемической неопределенностей.

In [ ]:
plt.figure(figsize=(6, 5))

plt.scatter(
    aleatoric_id,
    epistemic_id,
    alpha=0.3,
    color="#3498db",
    label="MNIST (ID)",
    s=15,
)
plt.scatter(
    aleatoric_ood,
    epistemic_ood,
    alpha=0.3,
    color="#e74c3c",
    label="Fashion-MNIST (OOD)",
    s=15,
)

max_val = max(np.max(aleatoric_ood), np.max(epistemic_ood))
plt.plot(
    [0, max_val],
    [0, max_val],
    "k--",
    alpha=0.5,
    label="Эпистемическая = Алеаторной",
)

plt.title("Соотношение типов неопределенности", fontsize=14)
plt.xlabel("Алеаторная", fontsize=13)
plt.ylabel("Эпистемическая", fontsize=13)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Оказывается, что почти всегда и для ID изображений, и для OOD преобладает алеаторная неопределенность, хотя по-хорошему мы ожидаем увидеть преобладание эпистемической для OOD наблюдений. Учитывая, что SWAG также не смог разделить типы неопределенности, возможно, дело в архитектуре модели, либо в специфике выбранных задач (алеаторная неопределенность высока из-за того, что каждая из сэмплированных моделей выдает примерно одинаковые распределения --> взаимная информация мала).